<a href="https://colab.research.google.com/github/emilieyds/aus-regional-income-indicators/blob/main/02_build_indicators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02. Build Regional Income Indicators

This notebook constructs regional income indicators from public ABS ASGS and LGA datasets.

The workflow:
1. Load public ABS regional datasets
2. Select income, population, household, and inequality variables
3. Standardize variable names
4. Reshape the data from long to wide format
5. Construct derived indicators such as income per person, income per household, and equivalised income
6. Save analysis-ready outputs

In [1]:
import os
import numpy as np
import pandas as pd

## 1) Load public ABS regional datasets

In [2]:
ASGS_URL = "https://api.data.abs.gov.au/files/ABS_ABS_REGIONAL_ASGS2021_1.0.0.csv"
LGA_URL = "https://api.data.abs.gov.au/files/ABS_ABS_REGIONAL_LGA2021_1.0.0.csv"

df_asgs = pd.read_csv(ASGS_URL, low_memory=False)
df_lga = pd.read_csv(LGA_URL, low_memory=False)

print("ASGS shape:", df_asgs.shape)
print("LGA shape:", df_lga.shape)

ASGS shape: (6033462, 24)
LGA shape: (1093239, 24)


## 2) Keep relevant geography levels

The ASGS dataset contains statistical geographies, while the LGA dataset contains Local Government Areas. These systems are kept as separate geography levels in the final dataset.

In [3]:
asgs_levels = ["AUS", "STE", "SA4", "SA3", "SA2"]

df_asgs_keep = df_asgs[df_asgs["REGIONTYPE"].isin(asgs_levels)].copy()
df_lga_keep = df_lga[df_lga["REGIONTYPE"] == "LGA2021"].copy()

print(df_asgs_keep["REGIONTYPE"].drop_duplicates().sort_values().tolist())
print(df_lga_keep["REGIONTYPE"].drop_duplicates().sort_values().tolist())

['AUS', 'SA2', 'SA3', 'SA4', 'STE']
['LGA2021']


## 3) Standardize geography identifiers

In [4]:
df_asgs_keep["geo_level"] = df_asgs_keep["REGIONTYPE"]
df_asgs_keep["geo_id"] = df_asgs_keep["ASGS_2021"].astype(str)
df_asgs_keep["geo_name"] = df_asgs_keep["Region"]

df_lga_keep["geo_level"] = df_lga_keep["REGIONTYPE"]
df_lga_keep["geo_id"] = df_lga_keep["LGA_2021"].astype(str).str.zfill(5)
df_lga_keep["geo_name"] = df_lga_keep["Region"]

df = pd.concat([df_asgs_keep, df_lga_keep], ignore_index=True)

df[["geo_level", "geo_id", "geo_name"]].head()

,geo_level,geo_id,geo_name
0,AUS,AUS,Australia
1,AUS,AUS,Australia
2,AUS,AUS,Australia
3,AUS,AUS,Australia
4,AUS,AUS,Australia


## 4) Select and rename ABS measures

In [5]:
income_map = {
    # Population and households
    "ERP_P_2": "NB_PERS_0_4",
    "ERP_P_3": "NB_PERS_5_9",
    "ERP_P_4": "NB_PERS_10_14",
    "ERP_P_20": "NB_PERS",
    "HHTYPE_5": "NB_HH",
    "HHTYPE_6": "NB_PERS_HH",

    # Income components
    "INCOME_3": "EMP_INC_TOT",
    "INCOME_6": "SELF_EMP_INC_TOT",
    "INCOME_9": "CAP_INC_TOT",

    # Total income
    "INCOME_19": "NB_INCOME_EARNERS",
    "INCOME_18": "TOT_INC",
    "INCOME_17": "TOT_INC_MEDIAN",
    "INCOME_36": "TOT_INC_MEAN",

    # Equivalised household income
    "EQUIV_2": "EQ_HH_INC_MEDIAN_WEEKLY",

    # Inequality
    "INCOME_41": "GINI",
    "INCOME_37": "P80_P20_RATIO",
}

df_income = df[df["MEASURE"].isin(income_map.keys())].copy()

df_income = df_income.rename(
    columns={
        "MEASURE": "measure",
        "TIME_PERIOD": "year",
        "OBS_VALUE": "value",
    }
)

df_income["variable"] = df_income["measure"].map(income_map)
df_income["year"] = df_income["year"].astype(int)

df_income[["geo_level", "geo_id", "geo_name", "year", "measure", "variable", "value"]].head()

,geo_level,geo_id,geo_name,year,measure,variable,value
0,AUS,AUS,Australia,2016,ERP_P_20,NB_PERS,24190907.0
1,AUS,AUS,Australia,2017,ERP_P_20,NB_PERS,24594202.0
2,AUS,AUS,Australia,2018,ERP_P_20,NB_PERS,24966643.0
3,AUS,AUS,Australia,2019,ERP_P_20,NB_PERS,25340217.0
4,AUS,AUS,Australia,2020,ERP_P_20,NB_PERS,25655289.0


## 5) Reshape data to wide format

In [6]:
wide = (
    df_income
    .groupby(["geo_level", "geo_id", "geo_name", "year", "variable"], as_index=False)["value"]
    .sum()
    .pivot_table(
        index=["geo_level", "geo_id", "geo_name", "year"],
        columns="variable",
        values="value",
        aggfunc="sum"
    )
    .reset_index()
)

wide.columns.name = None

print(wide.shape)
wide.head()

(23805, 20)


,geo_level,geo_id,geo_name,year,CAP_INC_TOT,EMP_INC_TOT,EQ_HH_INC_MEDIAN_WEEKLY,GINI,NB_HH,NB_INCOME_EARNERS,NB_PERS,NB_PERS_0_4,NB_PERS_10_14,NB_PERS_5_9,NB_PERS_HH,P80_P20_RATIO,SELF_EMP_INC_TOT,TOT_INC,TOT_INC_MEAN,TOT_INC_MEDIAN
0,AUS,AUS,Australia,2011,NaN,NaN,763.0,NaN,7760317.0,NaN,NaN,NaN,NaN,NaN,2.6,NaN,NaN,NaN,NaN,NaN
1,AUS,AUS,Australia,2016,81596.8,724934.7,877.0,0.484,8286079.0,13358252.0,24190907.0,1573626.0,1431690.0,1567281.0,2.6,4.99,50551.6,827875.1,61975.0,47692.0
2,AUS,AUS,Australia,2017,82573.5,746501.2,NaN,0.482,NaN,13678024.0,24594202.0,1574570.0,1473503.0,1588064.0,NaN,5.02,53168.9,856159.5,62594.0,48360.0
3,AUS,AUS,Australia,2018,88865.5,787316.7,NaN,0.483,NaN,14069082.0,24966643.0,1563728.0,1517847.0,1604857.0,NaN,5.02,53870.0,903888.7,64246.0,49805.0
4,AUS,AUS,Australia,2019,95562.1,828414.6,NaN,0.481,NaN,14425037.0,25340217.0,1554538.0,1561095.0,1616624.0,NaN,4.97,54149.9,951373.4,65953.0,51389.0


## 6) Convert total income variables from millions to dollars

ABS reports total income variables in millions of Australian dollars. These are converted to dollars before constructing per-person and per-household indicators.

In [7]:
income_total_cols = ["EMP_INC_TOT", "SELF_EMP_INC_TOT", "CAP_INC_TOT", "TOT_INC"]

for col in income_total_cols:
    if col in wide.columns:
        wide[col] = wide[col] * 1_000_000

## 7) Construct income indicators

In [8]:
wide["TOT_INC_PER_PERSON"] = wide["TOT_INC"] / wide["NB_PERS"]
wide["TOT_INC_PER_HH"] = wide["TOT_INC"] / wide["NB_HH"]

wide["EMP_INC_PER_PERSON"] = wide["EMP_INC_TOT"] / wide["NB_PERS"]
wide["SELF_EMP_INC_PER_PERSON"] = wide["SELF_EMP_INC_TOT"] / wide["NB_PERS"]
wide["CAP_INC_PER_PERSON"] = wide["CAP_INC_TOT"] / wide["NB_PERS"]

wide["EMP_INC_PER_HH"] = wide["EMP_INC_TOT"] / wide["NB_HH"]
wide["SELF_EMP_INC_PER_HH"] = wide["SELF_EMP_INC_TOT"] / wide["NB_HH"]
wide["CAP_INC_PER_HH"] = wide["CAP_INC_TOT"] / wide["NB_HH"]

wide[[
    "geo_level", "geo_id", "geo_name", "year",
    "TOT_INC_PER_PERSON", "TOT_INC_PER_HH",
    "EMP_INC_PER_PERSON", "SELF_EMP_INC_PER_PERSON", "CAP_INC_PER_PERSON"
]].head()

,geo_level,geo_id,geo_name,year,TOT_INC_PER_PERSON,TOT_INC_PER_HH,EMP_INC_PER_PERSON,SELF_EMP_INC_PER_PERSON,CAP_INC_PER_PERSON
0,AUS,AUS,Australia,2011,NaN,NaN,NaN,NaN,NaN
1,AUS,AUS,Australia,2016,34222.573796,99911.562513,29967.239343,2089.694281,3373.035992
2,AUS,AUS,Australia,2017,34811.436452,NaN,30352.731103,2161.846926,3357.437659
3,AUS,AUS,Australia,2018,36203.854078,NaN,31534.744178,2157.678948,3559.369195
4,AUS,AUS,Australia,2019,37544.011561,NaN,32691.693208,2136.915402,3771.163443


## 8) Construct equivalised income indicators

The ABS variable `EQUIV_2` reports median equivalised total household income on a weekly basis. This is converted to an annual value. A simple square-root equivalence scale is also used to construct an approximate equivalised mean income from household mean income.

In [9]:
wide["EQ_HH_INC_MEDIAN_ANNUAL"] = wide["EQ_HH_INC_MEDIAN_WEEKLY"] * 365.5 / 7

wide["TOT_INC_EQ_MEAN"] = wide["TOT_INC_PER_HH"] / np.sqrt(wide["NB_PERS_HH"])

wide[[
    "geo_level", "geo_id", "geo_name", "year",
    "EQ_HH_INC_MEDIAN_WEEKLY",
    "EQ_HH_INC_MEDIAN_ANNUAL",
    "TOT_INC_EQ_MEAN"
]].head()

,geo_level,geo_id,geo_name,year,EQ_HH_INC_MEDIAN_WEEKLY,EQ_HH_INC_MEDIAN_ANNUAL,TOT_INC_EQ_MEAN
0,AUS,AUS,Australia,2011,763.0,39839.500000,NaN
1,AUS,AUS,Australia,2016,877.0,45791.928571,61962.520694
2,AUS,AUS,Australia,2017,NaN,NaN,NaN
3,AUS,AUS,Australia,2018,NaN,NaN,NaN
4,AUS,AUS,Australia,2019,NaN,NaN,NaN


## 9) Add ASGS parent geography identifiers

STE, SA4, SA3, and SA2 codes are hierarchical. SA4 uses a 3-digit code, SA3 uses a 5-digit code, and SA2 uses a 9-digit code. These codes allow SA2 units to be linked upward to SA3, SA4, and State/Territory. GCCSA uses a separate 5-character alphanumeric code and is treated separately from the SA2–SA3–SA4 hierarchy.

In [10]:
wide["state_code"] = np.nan
wide["sa4_code"] = np.nan
wide["sa3_code"] = np.nan
wide["sa2_code"] = np.nan
wide["gccsa_code"] = np.nan

is_sa4 = wide["geo_level"] == "SA4"
is_sa3 = wide["geo_level"] == "SA3"
is_sa2 = wide["geo_level"] == "SA2"
is_gccsa = wide["geo_level"] == "GCCSA"

wide.loc[is_sa4, "geo_id_padded"] = wide.loc[is_sa4, "geo_id"].astype(str).str.zfill(3)
wide.loc[is_sa4, "state_code"] = wide.loc[is_sa4, "geo_id_padded"].str[0]
wide.loc[is_sa4, "sa4_code"] = wide.loc[is_sa4, "geo_id_padded"]

wide.loc[is_sa3, "geo_id_padded"] = wide.loc[is_sa3, "geo_id"].astype(str).str.zfill(5)
wide.loc[is_sa3, "state_code"] = wide.loc[is_sa3, "geo_id_padded"].str[0]
wide.loc[is_sa3, "sa4_code"] = wide.loc[is_sa3, "geo_id_padded"].str[:3]
wide.loc[is_sa3, "sa3_code"] = wide.loc[is_sa3, "geo_id_padded"]

wide.loc[is_sa2, "geo_id_padded"] = wide.loc[is_sa2, "geo_id"].astype(str).str.zfill(9)
wide.loc[is_sa2, "state_code"] = wide.loc[is_sa2, "geo_id_padded"].str[0]
wide.loc[is_sa2, "sa4_code"] = wide.loc[is_sa2, "geo_id_padded"].str[:3]
wide.loc[is_sa2, "sa3_code"] = wide.loc[is_sa2, "geo_id_padded"].str[:5]
wide.loc[is_sa2, "sa2_code"] = wide.loc[is_sa2, "geo_id_padded"]

wide.loc[is_gccsa, "geo_id_padded"] = wide.loc[is_gccsa, "geo_id"].astype(str)
wide.loc[is_gccsa, "state_code"] = wide.loc[is_gccsa, "geo_id_padded"].str[0]
wide.loc[is_gccsa, "gccsa_code"] = wide.loc[is_gccsa, "geo_id_padded"]

wide = wide.drop(columns=["geo_id_padded"])

/tmp/ipykernel_47758/1636763918.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1'
 '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1'
 '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1'
 '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1'
 '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1'
 '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1'
 '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1'
 '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1'
 '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1'
 '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1'
 '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '1' '2' '2'
 '2' '2' '2' '2' '2' '2' '2' 

## 10) Save analysis-ready outputs

In [11]:
os.makedirs("output", exist_ok=True)

wide.to_csv("output/australia_regional_income_indicators.csv", index=False)
wide.to_parquet("output/australia_regional_income_indicators.parquet", index=False)

print("Saved outputs:")
print("output/australia_regional_income_indicators.csv")
print("output/australia_regional_income_indicators.parquet")

Saved outputs:
output/australia_regional_income_indicators.csv
output/australia_regional_income_indicators.parquet


## Summary

This notebook constructs analysis-ready regional income indicators from public ABS ASGS and LGA datasets. The final output contains standardized geography identifiers, cleaned income and population variables, and derived indicators such as income per person, income per household, equivalised income, and inequality measures.